In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
import os
import ast
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from scipy.sparse import hstack
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env file

True

In [2]:
# --- Configuration ---
INPUT_CSV = Path("../data/clean_sample_for_embeddings_og.csv")
OUTPUT_CSV = Path("../data/clean_sample_with_imputed_prices.csv")
N_NEIGHBORS = 5 # Number of neighbors for KNN Imputation


In [11]:

# --- Step 1: Load Data ---
print(f"Loading data from {INPUT_CSV}...")
try:
    # Use the content ID to ensure access to the uploaded file
    df = pd.read_csv(INPUT_CSV)
except Exception as e:
    print(f"Error loading CSV: {e}")
    # Fallback/Exit strategy for file access issues
    print("Please ensure the CSV file is accessible in the current execution environment.")
    exit()



Loading data from ..\data\clean_sample_for_embeddings_og.csv...


In [12]:
# --- Step 2: Prepare Target and Split Data ---

# Rows with known price (Training Set)
train_df = df[df['price_clean'].notna()].copy()
# Rows with missing price (Imputation Set)
impute_df = df[df['price_clean'].isna()].copy()

print(f"\nTotal rows: {len(df)}")
print(f"Rows with known price (Train): {len(train_df)}")
print(f"Rows with missing price (Impute): {len(impute_df)}")

if len(impute_df) == 0:
    print("No missing prices (NaN) to impute. Saving the original file to a new name.")
    df.to_csv(OUTPUT_CSV, index=False)
    exit()

if len(train_df) == 0:
    print("No known prices available to train the model. Cannot impute.")
    exit()


# Target variable
y_train = train_df['price_clean'].values




Total rows: 312
Rows with known price (Train): 215
Rows with missing price (Impute): 97


In [13]:
# # --- Step 3: Feature Engineering and Preprocessing ---

# # A. Categorical Feature Preprocessing (material, color)
# # Using SimpleImputer to fill NaN strings with 'unknown'
# cat_imputer = Pipeline([
#     ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
#     ('onehot', OneHotEncoder(handle_unknown='ignore'))
# ])

# # B. Text Feature Preprocessing (combined_text)
# # Using TF-IDF to convert text into a sparse numerical matrix
# tfidf_vectorizer = TfidfVectorizer(max_features=500, stop_words='english')

# # C. Multi-Label Feature Preprocessing (category_list)
# # This requires manual processing as ColumnTransformer doesn't support MultiLabelBinarizer directly
# def preprocess_multilabel(df_series):
#     # Safely evaluate string representation of lists
#     list_of_lists = [ast.literal_eval(x) if pd.notna(x) and x.strip() else [] 
#                      for x in df_series]
    
#     # Fit on all categories across the entire dataset for consistent columns
#     mlb = MultiLabelBinarizer()
#     encoded_features = mlb.fit_transform(list_of_lists)
#     return encoded_features, mlb.classes_


# # Define the column transformer for simple categorical and text features
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('cat', cat_imputer, ['material', 'color']),
#         ('text', tfidf_vectorizer, 'combined_text')
#     ],
#     remainder='passthrough'
# )



In [14]:
# --- Step 3: Feature Engineering and Preprocessing (CORRECTED) ---
from sklearn.compose import make_column_selector as selector 
# Assuming 'category_list' and 'price' are the only non-processed columns 
# besides 'material', 'color', 'combined_text'

# A. Categorical Feature Preprocessing (material, color)
cat_imputer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)) # Ensure sparse output
])

# B. Text Feature Preprocessing (combined_text)
tfidf_vectorizer = TfidfVectorizer(max_features=500, stop_words='english')

# Define the column transformer for simple categorical and text features
# IMPORTANT: We explicitly list the feature columns we want to transform.
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', cat_imputer, ['material', 'color']),
        ('text', tfidf_vectorizer, 'combined_text')
    ],
    # FIX: Use 'drop' to prevent unlisted non-numeric columns 
    # (like 'category_list' or 'price') from being passed through.
    remainder='drop' 
)

# C. Multi-Label Feature Preprocessing (category_list)
# This requires manual processing as ColumnTransformer doesn't support MultiLabelBinarizer directly
def preprocess_multilabel(df_series):
    # Safely evaluate string representation of lists
    list_of_lists = [ast.literal_eval(x) if pd.notna(x) and x.strip() else [] 
                     for x in df_series]
    
    # Fit on all categories across the entire dataset for consistent columns
    mlb = MultiLabelBinarizer()
    encoded_features = mlb.fit_transform(list_of_lists)
    return encoded_features, mlb.classes_

In [15]:
# # --- Step 4: Fit and Transform Features ---

# print("Processing features (TF-IDF and One-Hot Encoding)...")

# # 1. Process Multi-Label Feature (category_list)
# cat_features_full, cat_classes = preprocess_multilabel(df['category_list'])

# # 2. Process all other features using the ColumnTransformer
# X_other_full = preprocessor.fit_transform(df)

# # 3. Combine all features (sparse matrix hstack)
# X_full = hstack([X_other_full, cat_features_full])

# # Split combined features back into Train and Impute sets
# X_train = X_full[train_df.index]
# X_impute = X_full[impute_df.index]




In [16]:
# --- Step 4: Fit and Transform Features (REVISED) ---
from scipy.sparse import hstack
import pandas as pd # Import necessary libraries if they weren't before

# 1. Prepare the DataFrame for processing
# Create a feature-only DataFrame, excluding the target and multi-label column
feature_cols = ['material', 'color', 'combined_text', 'category_list']
df_features = df[feature_cols]

print("Processing features (TF-IDF and One-Hot Encoding)...")

# 2. Process Multi-Label Feature (category_list)
# Pass only the specific column needed for MultiLabelBinarizer
cat_features_full, cat_classes = preprocess_multilabel(df_features['category_list'])

# 3. Process all other features using the ColumnTransformer
# Pass the full feature DataFrame (which now includes 'category_list')
# The ColumnTransformer is set to remainder='drop' and only selects its 
# specified columns ('material', 'color', 'combined_text').
X_other_full = preprocessor.fit_transform(df_features)

# 4. Combine all features (sparse matrix hstack)
# Both X_other_full and cat_features_full are sparse, so hstack works.
X_full = hstack([X_other_full, cat_features_full])
X_full_csr = X_full.tocsr()
# Split combined features back into Train and Impute sets
# Ensure train_df and impute_df indices are still valid for slicing X_full
X_train = X_full_csr[train_df.index]
X_impute = X_full_csr[impute_df.index]

print(f"X_full shape: {X_full.shape}")
print(f"X_train shape: {X_train.shape}")
print(f"X_impute shape: {X_impute.shape}")
print("Feature processing complete. Ready for KNN Imputation.")

Processing features (TF-IDF and One-Hot Encoding)...
X_full shape: (312, 866)
X_train shape: (215, 866)
X_impute shape: (97, 866)
Feature processing complete. Ready for KNN Imputation.


In [18]:
# --- Step 5: Train KNN Regressor and Predict ---

print(f"Training KNeighborsRegressor (K={N_NEIGHBORS})...")
knn_regressor = KNeighborsRegressor(n_neighbors=N_NEIGHBORS, weights='distance')

# Train the model on known prices
knn_regressor.fit(X_train, y_train)

# Predict missing prices
print("Predicting missing prices...")
predicted_prices = knn_regressor.predict(X_impute)
print(predicted_prices)
print("Prediction complete.")


Training KNeighborsRegressor (K=5)...
Predicting missing prices...
[106.83610435  66.20322317 106.83610435  66.20322317 129.14281669
  69.6120093   41.36093741  64.73426331  20.00504788  54.28053988
  20.39225323  73.04808897  66.36167375 149.25527263  54.85531577
  54.62547475  60.31034942  71.28352504  32.85920325  64.64970181
  83.89453952  88.10886116  61.27783631  79.00773214 119.12462939
  73.82461979 122.17505837  58.19911716  15.65686323 132.91026484
  64.34566013  68.33285048 111.01200704  61.57037596  72.10412795
  74.81492907  58.6264396  192.11222899  87.01775933  44.30689994
  75.65316029  17.80207302  38.45293792  60.48672092  32.20644716
 122.63967376  64.32774989  72.89141836  52.85464535  72.73615976
 156.06587215 122.31427437  54.87412915 117.34699043  60.23892708
  70.62978036  93.68664041  51.97303602 156.33473024 133.67268514
  21.00651923  77.99237639 190.71096596  60.72578899  66.26488276
  70.45754755  68.09134159  47.66938082  33.10159914 113.22293956
  83.8131

In [19]:
# --- Step 6: Final Update and Save ---

# Ensure predicted prices are positive and fill the NaN values
# Use a minimum price of 0.01 to avoid invalid values
predicted_prices = np.maximum(predicted_prices, 0.01)
df.loc[impute_df.index, 'price_clean'] = predicted_prices

print("\n--- Imputation Complete ---")
print(f"Successfully imputed {len(predicted_prices)} missing prices.")

# Save the updated DataFrame to a new CSV file
df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Cleaned DataFrame saved to: {OUTPUT_CSV}")
print("You can now use this file to re-generate your embeddings with accurate price information in the metadata!")



--- Imputation Complete ---
Successfully imputed 97 missing prices.

✅ Cleaned DataFrame saved to: ..\data\clean_sample_with_imputed_prices.csv
You can now use this file to re-generate your embeddings with accurate price information in the metadata!
